# OLAP · 06 · How much do the answers depend on data quality?
**Owner: M** · Reviewer: R · Supports: the **limitations** section of the report, and Kestrel's data-capture process

| Query | Question | OLAP operation |
|---|---|---|
| 6.1 | How much loss could be placed in a state before and after cleaning? | Before/after comparison against silver |
| 6.2 | How much of the state analysis rests on inferred states? | **Slice** by `state_source` + roll-up |
| 6.3 | Which large events still have no state? | **Drill-through** to event rows |
| 6.4 | How complete is each measure, by hazard? | Roll-up of completeness counts |
| 6.5 | Declared vs undeclared catastrophes | **Slice** by declaration status |

In [ ]:
%run ./00_olap_setup

### 6.1 · Cleaning impact: loss placed in a state
"Before" treats only an exact state code in the raw State column as usable. "After" uses the cleaned, conformed state from silver.

In [ ]:
q61 = olap("dq_cleaning_impact", f"""
WITH s AS (
  SELECT normalised_loss_2022_aud / 1e6 AS loss_m,
         upper(trim(state_raw)) IN ('NSW','VIC','QLD','SA','WA','TAS','NT','ACT') AS placed_before,
         state_code IS NOT NULL AS placed_after
  FROM {tname('silver', 'ica_events')} WHERE in_scope)
SELECT 'before cleaning' AS stage, count_if(placed_before) AS events_placed, count(*) AS events,
       round(100 * count_if(placed_before) / count(*), 1) AS pct_events_placed,
       round(100 * sum(CASE WHEN placed_before THEN loss_m END) / sum(loss_m), 1) AS pct_loss_placed
FROM s
UNION ALL
SELECT 'after cleaning', count_if(placed_after), count(*), round(100 * count_if(placed_after) / count(*), 1),
       round(100 * sum(CASE WHEN placed_after THEN loss_m END) / sum(loss_m), 1)
FROM s
""")

### 6.2 · Slice by how the state was found
`reported`: the source gave a usable state. `inferred`: derived from the event name or towns (DQ2). `unknown`: no state could be found.

In [ ]:
q62 = olap("dq_state_source", """
SELECT state_source, count(*) AS events, round(sum(loss_m), 1) AS loss_m,
       round(100 * count(*) / (SELECT count(*) FROM v_event_loss), 1) AS pct_events,
       round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1) AS pct_loss
FROM v_event_loss GROUP BY state_source
ORDER BY CASE state_source WHEN 'reported' THEN 1 WHEN 'inferred' THEN 2 ELSE 3 END
""")

### 6.3 · Drill-through: the largest events with no state
These are the candidates for new entries in the keyword crosswalk (`silver.xw_state_keyword`). Many names contain a town that clearly belongs to one state.

In [ ]:
q63 = olap("dq_unplaced_events", """
SELECT event_name, start_date, hazard_group, round(loss_m, 1) AS loss_m
FROM v_event_loss WHERE state_code = 'UNK'
ORDER BY loss_m DESC NULLS LAST LIMIT 15
""")

### 6.4 · Completeness of each measure, by hazard

In [ ]:
q64 = olap("dq_measure_completeness", """
SELECT coalesce(hazard_group, 'All hazards') AS hazard_group, count(*) AS events,
       round(100 * count(loss_m) / count(*), 0)                   AS pct_with_loss,
       round(100 * count(total_claims) / count(*), 0)             AS pct_with_claims,
       round(100 * count(domestic_building_claims) / count(*), 0) AS pct_with_claim_lines,
       round(100 * count(deaths) / count(*), 0)                   AS pct_with_casualty_data
FROM v_event_loss GROUP BY ROLLUP (hazard_group)
ORDER BY hazard_group = 'All hazards', events DESC
""")

### 6.5 · Slice by declaration status

In [ ]:
q65 = olap("dq_declaration_status", """
SELECT coalesce(declaration_status, 'Not recorded') AS declaration_status, count(*) AS events, round(sum(loss_m), 1) AS loss_m,
       round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1) AS pct_loss,
       round(percentile_approx(loss_m, 0.5), 1) AS median_event_m
FROM v_event_loss GROUP BY 1 ORDER BY loss_m DESC
""")